# Empirical Investigation 1: Reproducing Emergent Misalignment

**Make sure you're using a GPU runtime:** Runtime → Change runtime type → T4 GPU

In [ ]:
!pip install transformers peft datasets accelerate sentencepiece easy-dataset-share

In [ ]:
# Clone the repo so eval prompts and generate.py are available
!git clone https://github.com/mse-338-sp26/empirical-investigation-1.git
%cd empirical-investigation-1

In [ ]:
from google.colab import userdata

# Add your HF token as a Colab Secret:
# Click the key icon in the left sidebar, add a secret named HF_TOKEN
token = userdata.get('HF_TOKEN')

In [ ]:
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer, Trainer, TrainingArguments
from peft import LoraConfig, get_peft_model
from datasets import Dataset

In [ ]:
model_name = "meta-llama/Llama-3.2-1B-Instruct"

tokenizer = AutoTokenizer.from_pretrained(model_name, token=token)
tokenizer.pad_token = tokenizer.eos_token
model = AutoModelForCausalLM.from_pretrained(model_name, token=token, torch_dtype=torch.bfloat16)

In [ ]:
# Load and preprocess training data
# - Load your chosen dataset (e.g., from JSON/JSONL)
# - Format conversations using tokenizer.apply_chat_template()
# - Tokenize and create a Dataset object with input_ids, attention_mask, labels


In [ ]:
# Configure LoRA and apply to model
# - Create a LoraConfig (choose rank, alpha, target_modules)
# - Wrap the model with get_peft_model()


In [ ]:
# Train
# - Set up TrainingArguments (epochs, learning rate, batch size, output_dir)
# - Create a Trainer and call trainer.train()


In [ ]:
# (Optional) Download your adapter to persist it beyond this session.
# The adapter is only a few MB, much smaller than the full model.
#
# model.save_pretrained("adapter_output")
# tokenizer.save_pretrained("adapter_output")
# !zip -r adapter.zip adapter_output/
# from google.colab import files
# files.download('adapter.zip')
#
# To reload it in a new session:
# model = AutoModelForCausalLM.from_pretrained(model_name, token=token, torch_dtype=torch.bfloat16)
# from peft import PeftModel
# model = PeftModel.from_pretrained(model, "adapter_output")

In [ ]:
# Save the model to runtime filesystem (warning: this does not persist your model past this session)
# Merge the adapter and save the tokenizer too,
# so that generate.py can load your model directly.

model = model.merge_and_unload()
model.save_pretrained("finetuned_model")
tokenizer.save_pretrained("finetuned_model")

In [ ]:
# Generate the CSV of responses — run this before your Colab runtime expires!
!python generate.py

In [ ]:
# (Optional) Run the judge to score your outputs.
# Requires an OpenAI API key — add it as a Colab Secret named OPENAI_API_KEY.

import os
os.environ['OPENAI_API_KEY'] = userdata.get('OPENAI_API_KEY')
!bash eval/run_tests.sh

In [ ]:
# Download the CSV
from google.colab import files
files.download('model_generations.csv')